# 112장 · 배포 설정의 포트·볼륨과 확인 절차

배포 설정에서 포트는 요청 경로를, 볼륨은 데이터가 남는 위치를 정합니다. 설정을 읽어 예상한 값과 실제 배포 뒤 확인할 값을 나눕니다.

배포 설정에서 포트와 데이터 볼륨을 읽고, 같은 후보를 재현하는 데 필요한 이미지 지문과 플랫폼을 확인합니다. 설정 검토와 실제 배포 결과를 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 후보판의 배포 조건과 미확인 값을 분리하기

111장의 로컬 회귀 결과를 읽고 배포 명세에 넣습니다. 어떤 이미지와 저장소로 실행할지 정해지지 않았으면 배포를 보류합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter111.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter112-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter111.json"
sample_file = here / "chapter112-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['regression'])

교재 예시 chapter112-start.json
{'spelling': [True, True], 'local_candidate_ok': True}


두 번째 줄은 읽은 파일의 `regression` 값입니다.

## 접속 경로와 볼륨 마운트 해석

배포 시안　서버 한 프로세스, JSON 자료 볼륨 p1-json, 호스트 127.0.0.1, 교육용 접근 주체, 이미지 변수명 P1_IMAGE. 실제 이미지 식별자·플랫폼은 미확인입니다. 아래의 고정된 주소는 같은 컴퓨터의 접속 범위를 뜻합니다. 누군가 밖에서 접속했다고 보고하려면 다른 환경에서 실제로 확인한 응답이 필요합니다.


### 셀 01 · 포트와 볼륨 설정 해석

제공 Compose 설정의 p1 항목에서 실제로 지정한 값 두 개만 읽습니다.


In [1]:
p1_service = {'ports': ['127.0.0.1:8104:8104'],
 'volumes': ['p1-json:/app/input106/project-ui/backend/runtime-data'],
 'pull_policy': 'never',
 'image': '${P1_IMAGE:?verified image ID or digest required}'}
print(p1_service["ports"][0])
print(p1_service["volumes"][0])

127.0.0.1:8104:8104
p1-json:/app/input106/project-ui/backend/runtime-data


로컬 8104 포트와 JSON 저장 디렉터리가 보입니다. 설정에 항목이 있다는 사실만으로 프로세스가 실행 중이거나 자료가 저장되었다고 볼 수 없습니다.

127.0.0.1 바인딩은 호스트 안에서 접속할 포트를, 볼륨 매핑은 컨테이너 교체 뒤에도 유지할 데이터 경로를 나타냅니다. 이 설정을 읽은 것만으로 운영 환경에 같은 이미지가 배포됐는지는 확인할 수 없습니다. 배포 기록에는 이미지 지문과 실제 실행 플랫폼이 필요합니다.


#### 볼륨 설정을 읽고 확인할 것

<mark>마운트 대상은 컨테이너 안에서 서비스가 실제로 기록하는 경로와 같아야 합니다.</mark> 포트로 접속되는지만 검사하면 재생성 때 데이터가 사라지는 문제를 놓칩니다. 팀 설정에서 저장 경로와 마운트 경로를 나란히 놓고 같은 레코드를 다시 읽으세요.

선수 개념 · 72장 Docker Compose 네트워크와 볼륨

#### 이미지 식별자가 없는 배포 후보는 어디에서 멈출까?

```mermaid
flowchart TD
  candidate["제공된 후보 읽기"] --> identity{"이미지 식별자와 플랫폼이 있는가?"}
  identity -->|없음| hold["현재 후보 · 확인 대기"]
  identity -->|있음| boot["담당자가 환경에서 기동"]
  boot --> health["상태 요청 후 Q01 재조회"]
```

이 노트북의 후보에는 이미지 식별자와 플랫폼이 모두 비어 있어 현재 경로는 확인 대기입니다. 오른쪽의 상태 요청과 재조회는 배포 담당자가 나중에 수행할 확인 절차입니다.

## 배포 전후 검사 항목과 기대값

준비 순서 예시　검증한 이미지 식별자와 플랫폼 기록 → 해당 복사본으로 상태 확인 요청 → Q01의 답·근거·세션 재조회 → 실패 시 새 쓰기를 멈추고 직전 검증 버전으로 복귀 → 동일 질문과 기존 자료 재조회. 이미지 되돌리기와 바뀐 자료 복구는 별도 작업입니다. 현재 예제에는 외부 응답과 실제 롤백 관찰값이 없습니다.


### 팀 참고 예시 · 미검증 설정 식별


In [2]:
candidate = {'image_digest': None, 'platform': None}
print([key for key, value in candidate.items() if value is None])

['image_digest', 'platform']


출력은 image_digest와 platform 두 항목입니다. 팀 메모 예시 — 어떤 이미지인지, 어느 플랫폼에서 확인했는지 모두 비어 있어 배포를 보류한다. 보류는 이 값을 읽고 내린 판단입니다.


## 배포 담당자가 직접 확인할 접속 경로와 데이터 위치

112장의 설정 시안은 교육용 API를 같은 컴퓨터에만 엽니다. 팀원이 자신의 노트북에서 접근할 수 있다고 보고하기 전에 호스트 주소, 컨테이너 포트, 저장 경로가 각각 무엇을 뜻하는지 설정값으로 복원합니다.

### 셀 04 · 설정에서 재조회할 위치 추출하기

수업에 제공한 IPv4 포트 매핑 한 건과 이름 있는 볼륨 한 건을 읽습니다. 컨테이너를 시작하거나 외부 자원을 만들지 않습니다.

In [3]:
from pathlib import PurePosixPath

bind_host, public_port, service_port = p1_service["ports"][0].split(":")
volume_name, mount_path = p1_service["volumes"][0].split(":", 1)
print(f"같은 컴퓨터의 확인 주소 http://{bind_host}:{public_port}/health")
print("컨테이너 포트", service_port)
print(volume_name, PurePosixPath(mount_path).name)

같은 컴퓨터의 확인 주소 http://127.0.0.1:8104/health
컨테이너 포트 8104
p1-json runtime-data


주소는 127.0.0.1의 8104이고 마지막 경로명은 runtime-data입니다. 이 주소를 다른 컴퓨터의 브라우저에 쓰면 그 컴퓨터 자신의 로컬 주소를 가리킵니다.

### 배포 후 결과를 채울 실행 카드

<table><thead><tr><th scope="col">시점</th><th scope="col">담당자의 행동</th><th scope="col">남길 근거</th></tr></thead><tbody><tr><td>이미지 준비</td><td>검증한 이미지 식별자와 플랫폼을 후보에 연결한다.</td><td>실제 사용한 식별자·플랫폼. 현재 후보의 값은 미확인</td></tr><tr><td>프로세스 기동 뒤</td><td>같은 호스트에서 /health를 열고 새 세션에서 Q01을 보낸다.</td><td>상태 응답, RULE-A 2.1절과 요청 ID</td></tr><tr><td>같은 볼륨으로 재기동 뒤</td><td>기존 세션·요청으로 기록을 다시 읽는다.</td><td>같은 소유자의 같은 요청·출처가 남아 있는지 대조</td></tr></tbody></table>

팀이 원격 시연을 택했다면 팀 밖에서 실제로 사용한 주소와 관찰 시각을 별도로 기록하세요. 설정만 정리한 날에는 실행 카드의 관찰란을 비워 둡니다.

<mark>프로세스가 응답하는지와 업무 데이터가 남는지는 별도로 확인합니다.</mark> /health가 응답하더라도 서비스가 기록하는 위치와 마운트 경로가 어긋나면 재기동 후 같은 요청을 읽지 못할 수 있습니다.

113장은 배포에 필요한 실제 데이터 대신 임시 사본으로 복구 훈련을 수행하고 복구 범위를 표시합니다.

## 팀 사례에 적용하기 · 후보판의 배포 조건과 미확인 값을 분리하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
deploy = {"local_candidate_ok": incoming["regression"]["local_candidate_ok"],
          "port": p1_service["ports"][0], "mount": p1_service["volumes"][0],
          "digest": candidate["image_digest"], "status": "미배포 예시"}
new = {**incoming, "deployment": deploy}
print(deploy["port"], deploy["digest"], deploy["status"])

127.0.0.1:8104:8104 None 미배포 예시


회귀를 통과해도 이미지 식별자가 비어 있으면 배포된 서비스라고 기록할 수 없습니다. 팀이 실제 배포했을 때만 digest와 재조회 증거를 기록하세요.

In [ ]:
output_file = project_root / "team-project/final/chapter112.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter112.json


113장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

113장에는 서비스 이미지 식별자와 별개로 자료 볼륨, 쓰기를 멈출 방법, 별도 복구 대상을 알려 주세요. 이 노트북은 설정 읽기만 수행하며 컨테이너 생성이나 공개 배포를 실행하지 않습니다.
